# Feature engineering

Building the feature matrix for the target, `work_interfere` (4
ordinal classes: Never, Rarely, Sometimes, Often). XGBoost needs
`num_class` set explicitly for multi-class objectives, that gets set to
match the actual 4 levels present rather than a larger, stale count,
which would be harmless to the math (XGBoost just never predicts the
extra classes) but wasteful. Country and the workplace-culture columns
(benefits, care_options, leave, etc.) get one-hot encoded, not
label-encoded, since nominal categories don't have a natural order.

In [1]:
import os
import pandas as pd
import numpy as np

df = pd.read_pickle('cleaned_mhealth.pkl')

target_map = {'Never': 0, 'Rarely': 1, 'Sometimes': 2, 'Often': 3}
y = df['work_interfere'].map(target_map)
y.value_counts().sort_index()

work_interfere
0    213
1    173
2    464
3    142
Name: count, dtype: int64

In [2]:
print(f'this dataset actually has {y.nunique()} classes, num_class needs to match that, not a higher stale count')

this dataset actually has 4 classes, num_class needs to match that, not a higher stale count


In [3]:
categorical_cols = ['Gender', 'self_employed', 'family_history', 'no_employees', 'remote_work',
                    'tech_company', 'benefits', 'care_options', 'wellness_program', 'seek_help',
                    'anonymity', 'leave', 'mental_health_consequence', 'phys_health_consequence',
                    'coworkers', 'supervisor', 'mental_health_interview', 'phys_health_interview',
                    'mental_vs_physical', 'obs_consequence']

top_countries = df.Country.value_counts().head(10).index
df['CountryGroup'] = df.Country.where(df.Country.isin(top_countries), 'Other')

feature_cols = ['Age'] + categorical_cols + ['CountryGroup']
X = pd.get_dummies(df[feature_cols], columns = categorical_cols + ['CountryGroup'])
X.shape

(992, 72)

In [4]:
from sklearn.feature_selection import mutual_info_classif

mi_scores_wrong = mutual_info_classif(X, y, random_state = 42)
pd.Series(mi_scores_wrong, index = X.columns).sort_values(ascending = False).head(5)

CountryGroup_Poland         0.090363
family_history_No           0.053143
leave_Somewhat difficult    0.034257
Gender_Trans                0.030047
family_history_Yes          0.029413
dtype: float64

`CountryGroup_Poland` comes out on top, by a wide margin, which
doesn't make sense on its face. It turns out to have only 7 respondents
in the entire dataset. `mutual_info_classif`'s default estimator treats
every column as continuous, using a k-nearest-neighbors approach that's
known to be unstable and biased upward for very sparse binary columns
like a rare one-hot dummy. Every feature here actually is a binary 0/1
indicator (or a genuinely numeric one, only `Age`), so the fix is
passing `discrete_features` to tell the estimator that explicitly.

In [5]:
mi_scores = mutual_info_classif(X, y, discrete_features = True, random_state = 42)
mi_ranking = pd.Series(mi_scores, index = X.columns).sort_values(ascending = False)
mi_ranking.head(12)

Age                      0.062001
family_history_Yes       0.045353
family_history_No        0.045353
care_options_Yes         0.013382
leave_Very difficult     0.012787
obs_consequence_No       0.011459
obs_consequence_Yes      0.011459
seek_help_No             0.010789
mental_vs_physical_No    0.010782
seek_help_Don't know     0.010200
Gender_Male              0.010075
benefits_Don't know      0.009687
dtype: float64

With the estimator told the truth about its own inputs, `Age` and `family_history` (both Yes and No sides) top the ranking instead, matching notebook 01's chi-square test. Workplace-support signals (`care_options`, `leave` difficulty, `obs_consequence`) follow behind at a smaller but real magnitude. `CountryGroup_Poland` drops out of the top ranks entirely.

In [6]:
import pickle
os.makedirs('data', exist_ok = True)
with open('data/model_matrix.pkl', 'wb') as f:
    pickle.dump({'X': X, 'y': y}, f)

import json
with open('outputs/feature_engineering_summary.json', 'w') as f:
    json.dump({
        'n_features': int(X.shape[1]),
        'n_classes': int(y.nunique()),
        'top_mi_feature': mi_ranking.index[0],
    }, f, indent = 2)
X.shape

(992, 72)